# KAT-VAD v2 — P3 / **E1: rate-matched DoTA evaluation** (no training)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§8 (Amendment 2, J1–J8) + §9 (Amendment 3: D12, J9′, J10)** · Plan: `.project/plans/katvad-v2-e0-e2.md` P3.
Re-scores **3 finished KIP-off checkpoints** on **DoTA-dev (702 clips)** from `DoTA_s1_ncc`: s2024 from phase 4, s2025 and
s2026 **retrained** by `colab/v2/p3_retrain_kipoff.ipynb` (phase 4's copies are mid-training snapshots — Amendment 3).

| arm | features | sequence |
|---|---|---|
| **A** | `s1[::8]` (0.8 s/step, today's protocol) | whole clip |
| **B** | `s1[::3]` (0.3 s/step, T2's rate) | whole clip |
| **C** | `s1[::3]` | windows W = 20, hop 4, overlap-averaged |

All arms are interpolated to native frames and scored against the same native labels. Decision (J7): B or C is adopted
only if its paired Δ vs A has a **cluster-bootstrap lower bound > 0**; otherwise A stays.

**Drive layout:**

| folder | holds | this notebook |
|---|---|---|
| `Thesis-V2/kat-vad/` | v2 code (upload the whole tree) | imports from here |
| `Thesis-V2/outputs/v2_p4_retrain/` | retrained s2025, s2026 | reads |
| `Thesis-V2/outputs/REPORTS/v2_E1/` | E1 read-out | **writes** |
| `Thesis/data/`, `Thesis/cache/` | data, CLIP caches (`DoTA_s1_ncc`, `DoTA_s8_ncc`) | reads |
| `Thesis/outputs/DADA2000_orig_phase4/s2024/` | the one finished phase-4 checkpoint | reads |

| step | what | GPU? |
|---|---|---|
| 0 | **run `p3_retrain_kipoff.ipynb` first** (until its §3 prints VERIFIED) | yes |
| 1 | preflight: code has Amendment 3, three checkpoints are finished (J10), caches exist | no |
| 2 | stage DoTA-dev s1 + s8 features to local disk | no |
| 3 | run E1 (J10 → A → **J9′ gate** → B, C → read-out) | yes (T4 is plenty) |
| 4 | record | no |

**Do not change a flag after reading §3.** The rule was committed before the first number. DoTA-eval is never loaded.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

# Drive layout (2026-09-29): v2 code + v2 outputs live in Thesis-V2; data, caches, checkpoints and the
# v1-era runs (phase 4) stay in Thesis and are only READ from there.
DRIVE    = '/content/drive/MyDrive/Thesis'        # shared: data/, cache/, ckpts/, outputs/ (v1 runs)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # v2: kat-vad/ (code), outputs/ (v2 runs + read-outs)
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'
V1_OUTPUTS = Path(DRIVE) / 'outputs'               # phase-4 runs (read-only)

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# core.constants reads KATVAD_* at IMPORT time. Re-running this cell in the same kernel would reuse the
# module imported under the old env, so drop every cached core.* module first.
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
for got, want in ((constants.DATA_ROOT, f'{DRIVE}/data'), (constants.CACHE_ROOT, f'{DRIVE}/cache'),
                  (constants.OUTPUT_ROOT, f'{DRIVE_V2}/outputs')):
    assert str(got) == want, f'{got} != {want}'

# D12 (Amendment 3): s2024 = phase 4's finished checkpoint; s2025/s2026 = retrained (p3_retrain_kipoff.ipynb).
CKPTS = {
    's2024': V1_OUTPUTS / 'DADA2000_orig_phase4' / 's2024' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2025': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2025' / 'stage2_kip_off' / 'checkpoint_last.pt',
    's2026': constants.OUTPUT_ROOT / 'v2_p4_retrain' / 's2026' / 'stage2_kip_off' / 'checkpoint_last.pt',
}
S1_DIR   = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
S8_DIR   = constants.CLIP_CACHE_DIR / 'DoTA_s8_ncc'
DOTA     = constants.DATA_ROOT / 'DoTA'
LABELS   = DOTA / 'labels_s8'                       # only defs.json is read from here
REPORT   = constants.OUTPUT_ROOT / 'REPORTS' / 'v2_E1'
LOCAL_S1 = Path('/content/e1/DoTA_s1_ncc_dev')      # VM-local disk, never Drive
LOCAL_S8 = Path('/content/e1/DoTA_s8_ncc_dev')
REPORT.mkdir(parents=True, exist_ok=True)
for name in ('REPO', 'S1_DIR', 'S8_DIR', 'DOTA', 'REPORT'):
    print(f'{name:7s}', globals()[name])
for name, path in CKPTS.items():
    print(f'{name:7s}', path)

In [ ]:
%%bash
# Same pins as phase_4.ipynb: rate_matched_eval imports core.train / core.inference, which pull in
# faiss (KNN cache), av (video io) and einops; transformers is PINNED because core/models/clip_text.py
# reaches into 4.x internals (lesson C7) -- v5 has no nested CLIPTextModel.text_model.
# Restart the runtime after this cell if pip printed a transformers downgrade, then re-run from cell 1.
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__, '| av/einops/faiss OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU (CPU works, ~10x slower)'

## 1. Preflight — refuse to start on a tree or data that E1 was not registered on

In [ ]:
import json
import subprocess

import torch

addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert 'Amendment 3' in addendum and 'J10' in addendum, (
    f'{REPO} predates Amendment 3 -- re-upload the v2 code there')
from core.tools import rate_matched_eval  # noqa: E402  (fails fast if absent)
from core.data.v2_splits import load_split  # noqa: E402

dev = load_split(constants.V2_SPLIT_DOTA_DEV)        # sha1-checked against the frozen manifest
assert len(dev) == 702, len(dev)

# J10 up front (the tool re-checks it): a mid-training snapshot must not reach the scorer.
RUN_ARGS = []
for name, ckpt in CKPTS.items():
    assert ckpt.is_file(), f'missing {ckpt} -- run p3_retrain_kipoff.ipynb first' if name != 's2024' else f'missing {ckpt}'
    step = int(torch.load(ckpt, map_location='cpu', weights_only=False).get('global_step', -1))
    last = rate_matched_eval.metrics_last_step(ckpt.parent / 'metrics.jsonl')
    assert step == last, f'J10: {name} checkpoint step {step} != metrics last step {last}'
    RUN_ARGS += ['--run', name, str(ckpt)]
    print(f'{name}: finished checkpoint, global_step {step}')

for f in (DOTA / 'metadata_val.json', DOTA / 'val_split.txt', LABELS / constants.DEFS_FILENAME):
    assert f.is_file(), f'missing {f}'
for d in (S1_DIR, S8_DIR):
    missing = [v for v in dev if not (d / f'{v}.npy').exists()]
    assert not missing, f'{len(missing)} DoTA-dev clips have no feature in {d}: {missing[:5]}'
print(f'DoTA-dev {len(dev)} clips, all present in {S1_DIR.name} and {S8_DIR.name}')

## 2. Stage DoTA-dev s1 + s8 features to local disk (only the 702 dev clips)

Drive FUSE reads are slow; staging makes each file one read. s8 is the J9′ reference (the evaluator's input path).

In [ ]:
import shutil

for src_dir, dst_dir in ((S1_DIR, LOCAL_S1), (S8_DIR, LOCAL_S8)):
    dst_dir.mkdir(parents=True, exist_ok=True)
    for v in dev:
        dst = dst_dir / f'{v}.npy'
        if not dst.exists():
            shutil.copy2(src_dir / f'{v}.npy', dst)
    print(f'staged {len(list(dst_dir.glob("*.npy")))} files -> {dst_dir}')

## 3. Run E1

Order inside the tool, per checkpoint: **J10** (finished checkpoint) → arm A → **J9′** (A's curve from `s1[::8]` must
equal the curve on the `DoTA_s8_ncc` cache on every step of every dev clip, atol 1e-4; a failure **raises and nothing
else is scored**) → then arms B, C for all checkpoints → read-out. Expect ~10–20 min on a T4.

If J9′ or J10 fails: **stop**. Do not loosen the tolerance; send the error to Claude.

In [ ]:
cmd = [sys.executable, '-m', 'core.tools.rate_matched_eval', *RUN_ARGS,
       '--s1-dir', LOCAL_S1, '--s8-dir', LOCAL_S8, '--metadata', DOTA / 'metadata_val.json',
       '--split-file', DOTA / 'val_split.txt', '--data-dir', LABELS,
       '--out-dir', REPORT, '--device', 'auto']
LOG = REPORT / 'e1_run.log'

# Stream the child's stdout+stderr into this cell AND a log file on Drive: Colab does not show a
# subprocess's output by itself, so a failure would otherwise surface only as "exit status 1".
tail = []
with subprocess.Popen([str(a) for a in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as proc, LOG.open('w') as log:
    for line in proc.stdout:
        log.write(line)
        tail = (tail + [line])[-40:]
        print(line, end='')
if proc.returncode:
    print('\n'.join(['', '=' * 30, f'E1 FAILED (exit {proc.returncode}); last lines -- paste these to Claude:', '=' * 30]))
    print(''.join(tail))
    raise RuntimeError(f'rate_matched_eval exited {proc.returncode}; full log: {LOG}')
print((REPORT / rate_matched_eval.READOUT_MD).read_text())

## 4. Record

`e1_readout.{json,md}` are on Drive under `Thesis-V2/outputs/REPORTS/v2_E1/`. Paste the markdown back to Claude. Then:

* **adopt A** → DoTA protocol stays s8 whole clip; P2 (E2) stands as is.
* **adopt B or C** → D11: E2(b) and E2(c) are re-read on DoTA-dev at stride 3 before P5 fixes the CRN reference
  (Claude ships that runbook next).